In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from scipy.spatial.distance import mahalanobis

## Question 1: ATE with inverse probability weighting (IPW)


In [2]:
df1 = pd.read_csv("homework_8.1.csv")

# Step 1: estimate propensity scores, P = Pr(X = 1 | Z)
model = LogisticRegression()
model.fit(df1[["Z"]], df1["X"])
df1["P"] = model.predict_proba(df1[["Z"]])[:, 1]   # column 1 = probability that X = 1

# Step 2: inverse probability weights
df1["weight"] = np.where(df1["X"] == 1, 1 / df1["P"], 1 / (1 - df1["P"]))

# Step 3: weighted mean of Y in each group, then the difference
treated = df1[df1["X"] == 1]
untreated = df1[df1["X"] == 0]
y1 = np.average(treated["Y"], weights=treated["weight"])
y0 = np.average(untreated["Y"], weights=untreated["weight"])
ate = y1 - y0

# For comparison: the naive difference with no weighting (biased by Z)
naive = treated["Y"].mean() - untreated["Y"].mean()

# Horvitz-Thompson version: divide by N instead of the sum of weights
X, Y, P = df1["X"], df1["Y"], df1["P"]
ate_ht = np.mean(X * Y / P) - np.mean((1 - X) * Y / (1 - P))

print(f"Naive difference (biased):   {naive:.3f}")
print(f"IPW ATE (weighted means):    {ate:.3f}")
print(f"IPW ATE (Horvitz-Thompson):  {ate_ht:.3f}")

Naive difference (biased):   3.037
IPW ATE (weighted means):    2.274
IPW ATE (Horvitz-Thompson):  2.239


## Question 2: Propensity scores of the first three rows

In [3]:
print("First three propensity scores:", df1["P"].head(3).round(2).tolist())
print(df1[["X", "Z", "P"]].head(3))

First three propensity scores: [0.84, 0.58, 0.71]
   X         Z         P
0  1  1.764052  0.840114
1  0  0.400157  0.584646
2  0  0.978738  0.711082


## Question 3: ATE with Mahalanobis matching

In [7]:
df2 = pd.read_csv("homework_8.2.csv")

# Inverse covariance matrix from ALL rows: 2 x N matrix -> 2 x 2 covariance -> invert
Z = np.vstack([df2["Z1"], df2["Z2"]])   # shape (2, N): each row is one variable
VI = np.linalg.inv(np.cov(Z))           # shape (2, 2)

treated2 = df2[df2["X"] == 1].reset_index(drop=True)
untreated2 = df2[df2["X"] == 0].reset_index(drop=True)
untreated_Z = untreated2[["Z1", "Z2"]].values

effects, match_dist, match_idx = [], [], []
for _, row in treated2.iterrows():
    t_point = row[["Z1", "Z2"]].values
    # Distance from this treated row to EVERY untreated row
    dists = np.array([mahalanobis(t_point, u_point, VI) for u_point in untreated_Z])
    best = dists.argmin()                       # nearest untreated row (with replacement)
    effects.append(row["Y"] - untreated2.loc[best, "Y"])
    match_dist.append(dists[best])
    match_idx.append(best)                      # remember WHICH untreated row was the match

treated2["effect"] = effects
treated2["match_dist"] = match_dist
treated2["match_idx"] = match_idx

print(f"Matched ATE: {np.mean(effects):.3f}")

Matched ATE: 3.438


## Question 4: The treated row with the least common support

In [8]:
worst = treated2.loc[treated2["match_dist"].idxmax()]
nearest = untreated2.loc[worst["match_idx"]]

print(f"Treated row with least common support: Z1 = {worst['Z1']:.2f}, Z2 = {worst['Z2']:.2f}")
print(f"Its nearest untreated match (ANSWER):  Z1 = {nearest['Z1']:.2f}, Z2 = {nearest['Z2']:.2f}")
print(f"Mahalanobis distance between them:     {worst['match_dist']:.3f}")

Treated row with least common support: Z1 = 2.70, Z2 = 0.54
Its nearest untreated match (ANSWER):  Z1 = 1.52, Z2 = -1.28
Mahalanobis distance between them:     1.383
